# Context Window: ChatGPT bhool kyun jaata hai?
**AI Explained · Episode 2 (full version)** · Jajati AI

Har cell ko upar se neeche chalao (Shift + Enter). Google Colab mein free chalta hai.

Tokens `tiktoken` se gine jaate hain (`o200k_base` = GPT-4o family), wahi tokenizer jo Episode 1 mein dekha.

In [1]:
!pip install -q tiktoken

In [2]:
import tiktoken
enc = tiktoken.get_encoding("o200k_base")

def count(text):
    return len(enc.encode(text))

print(count("Mera naam Riya hai"), "tokens")

6 tokens


In [3]:
chat = [
    "Riya: Hi! Mera naam Riya hai, Pune se.",
    "AI: Hi Riya! Batao, kya madad karun?",
    "Riya: Python seekhna hai, roz 1 ghanta.",
    "AI: Week 1: variables, loops, functions.",
    "Riya: Loops samajh nahi aate, example do.",
    "AI: for i in range(3): print(i)",
]
for m in chat:
    print(count(m), m)
print("total:", sum(count(m) for m in chat))

15 Riya: Hi! Mera naam Riya hai, Pune se.
15 AI: Hi Riya! Batao, kya madad karun?
14 Riya: Python seekhna hai, roz 1 ghanta.
12 AI: Week 1: variables, loops, functions.
15 Riya: Loops samajh nahi aate, example do.
12 AI: for i in range(3): print(i)
total: 83


In [4]:
sent = 0
for turn in range(1, len(chat) + 1):
    n = sum(count(m) for m in chat[:turn])
    sent += n
    print("turn", turn, "-> AI padhta hai", n)
print("kul padhe:", sent, "tokens")

turn 1 -> AI padhta hai 15
turn 2 -> AI padhta hai 30
turn 3 -> AI padhta hai 44
turn 4 -> AI padhta hai 56
turn 5 -> AI padhta hai 71
turn 6 -> AI padhta hai 83
kul padhe: 299 tokens


In [5]:
def desk(chat, limit):
    kept, used = [], 0
    for m in reversed(chat):
        if used + count(m) > limit:
            break
        kept.insert(0, m)
        used += count(m)
    return kept

In [6]:
chat.append("Riya: Mera naam kya hai?")
d = desk(chat, 60)
for m in d:
    print(m)
print("Naam desk par hai?",
      any("Riya hai" in m for m in d))

AI: Week 1: variables, loops, functions.
Riya: Loops samajh nahi aate, example do.
AI: for i in range(3): print(i)
Riya: Mera naam kya hai?
Naam desk par hai? False


In [7]:
LIMIT = 128_000   # GPT-4o ki desk (tokens)
en = ("Tomorrow is my exam. Please make a short "
      "study plan for maths and science.")
hi = ("कल मेरी परीक्षा है। कृपया गणित और "
      "विज्ञान का छोटा सा study plan बनाओ।")
for name, t in (("English", en), ("Hindi", hi)):
    per_word = count(t) / len(t.split())
    words = int(LIMIT / per_word)
    print(name, words, "words =", words // 300,
          "pages")

English 112000 words = 373 pages
Hindi 85333 words = 284 pages


In [8]:
for n in (1_000, 2_000, 4_000, 8_000):
    print(f"{n:,} tokens -> {n * n:,} jode")

1,000 tokens -> 1,000,000 jode
2,000 tokens -> 4,000,000 jode
4,000 tokens -> 16,000,000 jode
8,000 tokens -> 64,000,000 jode


## Lost in the Middle
Liu et al. (2023): lambe input mein jawab shuru ya aakhir mein ho, to models use behtar dhoondhte hain; beech mein ho, to kam. Is notebook mein model nahi hai, isliye ye research se hai, code se nahi: https://arxiv.org/abs/2307.03172

In [9]:
summary = ("Riya, Pune. Python seekh rahi hai, "
           "roz 1 ghanta. Loops mein help chahiye.")
full = sum(count(m) for m in chat)
print(full, "->", count(summary), "tokens")

92 -> 26 tokens


In [10]:
# Apna message daalo aur chalao:
msg = "Mujhe AI ke baare mein sab seekhna hai"
print(count(msg), "tokens")
print(LIMIT // count(msg), "messages ek desk par")

12 tokens
10666 messages ek desk par
